### Azeotrope verification using unifac

#### objective
The verification of the presence/absence of an azeotrope in a binary mixture

we will consider the acetone-methanol binary system

In [ ]:
# import packages
from src.unifac_utils import load_gc, prop_calc, pyo_solve_antoine, pyo_unifac_bubbleT, pyo_unifac_azeotrope
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
# load the data
file_path = '../data/unifac.xlsx'
df_gc, df_a = load_gc(file_path, output_format='dataframe')

In [ ]:
# define the binary system:
# dict_binary = {'CH3': [1, 0],
#                'CH3CO': [1, 0],
#                'CH3OH': [0, 1]}
# dict_binary = {'CH3OH': [1, 0],
#                'H2O': [0, 1]}
dict_binary = {'CH3': [1, 0],
               'CH3CO': [1, 0],
               'H2O': [0, 1]}
df_binary = pd.DataFrame(dict_binary)

In [ ]:
# calculate properties
df_props = prop_calc(df_binary, ['tb', 'tc', 'pc', 'acc'], df_gc)
# if acetone
df_props['tb'][0] = 329.15
df_props['tc'][0] = 508.20
# convert pc to MPa
df_props['pc'] = df_props['pc'] / 10
# solve for the Antoine parameters
dict_antoine = {'A': {}, 'B': {}, 'C': {}}
for i in df_binary.index:
    tb = df_props['tb'].iloc[i]
    tc = df_props['tc'].iloc[i]
    pc = df_props['pc'].iloc[i]
    p0 = 0.101325
    w = df_props['acc'].iloc[i]
    output = pyo_solve_antoine(tb, tc, pc, p0, w, verbose=False)
    dict_antoine['A'][i] = output['A']
    dict_antoine['B'][i] = output['B']
    dict_antoine['C'][i] = output['C']
df_antoine = pd.DataFrame(dict_antoine)
array_antoine = np.array(df_antoine)

In [ ]:
# define the total pressure
P = 0.101325  # atmospheric in MPa
# define the number of composition points to be evaluated
n_data = 10
# generate the compositions
x0 = np.stack((np.linspace(1, 0, n_data), 1 - np.linspace(1, 0, n_data)), axis=1)
# generate initial guess for the temperature
T0 = np.linspace(df_props['tb'][0], df_props['tb'][1], n_data).flatten()

In [ ]:
solution, model = pyo_unifac_bubbleT(x0, T0, P, array_antoine, df_binary)

In [ ]:
# retrieving the solutions
y = np.empty_like(x0)
gamma = np.empty_like(x0)
T = np.empty_like(T0)
psat = np.empty_like(x0)

for i in model.y.keys():
    y[i] = model.y[i].value

for i in model.gamma.keys():
    gamma[i] = model.gamma[i].value

for i in model.T.keys():
    T[i] = model.T[i].value

for i in model.psat.keys():
    psat[i] = model.psat[i].value

We plot the vle diagram

In [ ]:
plt.plot(x0[:, 0], y[:, 0])
plt.title('Acetone + water')
plt.plot([0, 1], [0, 1])
plt.xlim((0.9, 1))
plt.ylim((0.9, 1))
plt.xlabel('x(acetone)')
plt.ylabel('y(acetone)')
plt.show()

we consider the relative volatility at the edges of the vle:
$$
\log \alpha_{A,B} \Biggr|_{x_B = 0} * \log \alpha_{A,B} \Biggr|_{x_B = 1}
$$
$$
\log \dfrac{K_A}{K_B} \Biggr|_{x_B = 0} * \log \dfrac{K_A}{K_B} \Biggr|_{x_B = 1}
$$
with
$$
K_A = \dfrac{y_A}{x_A}
$$
as such, this can be simplified to the following
$$
\log \dfrac{1}{K_B} \Biggr|_{x_B = 0} * \log \dfrac{K_A}{1} \Biggr|_{x_B = 1}
$$
as such, this can be simplified to the following
$$
\log \dfrac{1}{\gamma_B P^*_B} \Biggr|_{x_B = 0} * \log \dfrac{\gamma_A P^*_A}{1} \Biggr|_{x_B = 1}
$$

In [ ]:
print('there is an azeotrope between A-B: ', np.log(gamma[n_data, 0]*psat[n_data, 0]/P) * - np.log(gamma[0, 1]*psat[0, 1]/P)<0)
print('there is an azeotrope between A-E: ', np.log(gamma[-1, 0]*psat[-1, 0]/P) * - np.log(gamma[0, 2]*psat[0, 2]/P)<0)
print('there is an azeotrope between B-E: ', np.log(gamma[-1, 1]*psat[-1, 1]/P) * - np.log(gamma[n_data, 2]*psat[n_data, 2]/P) <0)

In [ ]:
print('there is an azeotrope: ',-np.log(gamma[-1,1]*psat[-1,1])*np.log(gamma[0,0]*psat[0,0]) <= 0)

In [ ]:
-np.log(gamma[-1,1]*psat[-1,1])*np.log(gamma[0,0]*psat[0,0])


In [ ]:
x0_az = np.array([0.5, 0.50]).reshape(1,2)
T0_az = df_props['tb'].mean()
az_solution, az_model = pyo_unifac_azeotrope(x0_az, T0_az, P, array_antoine, df_binary)

In [ ]:
# retrieving the solutions
az_x = np.empty_like(x0_az)
az_y = np.empty_like(x0_az)
az_gamma = np.empty_like(x0_az)
az_psat = np.empty_like(x0_az)

for i in az_model.y.keys():
    az_y[i] = az_model.y[i].value

for i in az_model.x.keys():
    az_x[i] = az_model.x[i].value

for i in az_model.gamma.keys():
    az_gamma[i] = az_model.gamma[i].value

for i in az_model.T.keys():
    az_T = az_model.T[i].value

for i in az_model.psat.keys():
    az_psat[i] = az_model.psat[i].value

In [ ]:
plt.plot(x0[:, 0], y[:, 0], 'b', linewidth=1.5)
plt.plot([0, 1], [0, 1], 'k', linewidth=1.5)
plt.plot(az_x[0,0], az_y[0,0], marker="o", markersize=20, markeredgecolor="red", markerfacecolor="None")
plt.grid()
plt.xlim(0.0, 1.0)
plt.ylim(0.0, 1.0)
plt.tight_layout()
plt.show()

In [ ]:
az_x

In [ ]:
az_y

In [ ]:
df_props